# Local run of the UK implementation of WOFOST 8.0  
***
**Author**: Mattia C. Mancini (m.c.mancini@exeter.ac.uk)  
**Affiliation**: LEEP Institute, University of Exeter  
**Date**: November 13th, 2025  
***  
This notebook illustrates how to run a local instance of the UK implementation of the WOFOST 8.0 crop yield model for a specified parcel, crop, and with default or user-modified input parameters.  
The notebook will go through the following steps:  
1. [Specify a location for which the model needs to be run.](#1-specify-a-location-for-which-the-model-needs-to-be-run)
2. [Generate an instance of the WOFOST simulator for that location.](#2-generate-an-instance-of-the-wofost-simulator-for-that-location)
3. [Specify a crop and its management](#3-specify-a-crop-and-its-management)
4. [Pass the crop and management to the simulator to return yields.](#4-pass-the-crop-and-management-to-the-simulator-to-return-yields)

## 1. Specify a location for which the model needs to be run
Locations in the UK implementation of WOFOST are parcels (fields) which can be identified with a *gid*, i.e., a unique integer identifier that comes from the vector land parcels in the [CEH Vector Land Cover Map](https://www.ceh.ac.uk/data/ukceh-land-cover-maps).  
Instantiating a parcel only requires to know the `gid` of the parcel of interest and is done using the class [`Parcel`](https://github.com/mcmancini/UkWofost/blob/dev/ukwofost/core/parcel.py)  
Printing the newly instantiated object returns some summary attributes of the parcel of interest including the Ordnance Survey code of its centroid and its average elevation.

In [ ]:
# Update sys path so notebook can access ukwofost package
import sys
sys.path.append('../')

In [ ]:
from ukwofost.core.parcel import Parcel
GID = 1491
parcel = Parcel(gid=GID)
print(parcel)

## 2. Generate an instance of the WOFOST simulator for that location
The instance of the class `Parcel` generated above contains all the information needed to generate an instance of the [`WofostSimulator`](https://github.com/mcmancini/UkWofost/blob/dev/ukwofost/core/simulation_manager.py). This is an object that, for the specified parcel, retrieves and stores all the necessary weather, soil and location specific data which will allow to drive the crop yield model. A number of weather data providers and soil data providers are available, but the defaults (and currently best) are [Mesoclim](https://github.com/ilyamaclean/mesoclim) for weather data and [SoilGrids](https://isric.org/explore/soilgrids) for soil data.  
Printing an instance of the `WofostSimulator` returns some location specific information.

In [ ]:
from ukwofost.core.simulation_manager import WofostSimulator

sim = WofostSimulator(
    location=parcel, weather_provider="Mesoclim", soil_provider="SoilGrids"
)

print(sim)

The data in the `WofostSimulator` is stored following the standard [pcse](https://github.com/ajwdewit/pcse) implementation.  
For example, weather data can be browsed as follows:

In [ ]:
print(sim.wdp)

Daily observations can be retrieved as follows:

In [ ]:
import datetime as dt

date = dt.date(2020, 6, 1)
daily_obs = sim.wdp(date)
print(daily_obs)

Equally, soil data can be retrieved as follows:

In [ ]:
print (sim.soildata)

## 3. Specify a crop and its management
One the `WofostSimulator` has been instantiated, the only thing left to do is to specify for which crop and under what crop management we want to return yields. This is done using the [`Crop`](https://github.com/mcmancini/UkWofost/blob/dev/ukwofost/core/crop_manager.py#L16) class for single crops, or a combination of the [`Crop`](https://github.com/mcmancini/UkWofost/blob/dev/ukwofost/core/crop_manager.py#L16) and [`CropRotation`](https://github.com/mcmancini/UkWofost/blob/dev/ukwofost/core/crop_manager.py#L288) classes. This allows to specify things like the year for which we need to run the crop yield model, the crop and variety of interest, the timing of fertilisation events, the quantity of fertiliser applied at each fertilisation event and so on. The minimum required information to build an instance of the `Crop` class is the year in which the crop is planted, the crop to plant and its agromanagement information. A default agromanagement is stored for simplicity, and can be overridden for custom management.  
A crop is instantiated as follows: 

In [ ]:
import copy
from ukwofost.core.crop_manager import Crop
from ukwofost.core.defaults import defaults

CROP = "winter_wheat"
YEAR = 2019
crop_management = copy.deepcopy(defaults.get("management").get(CROP))
crop = Crop(calendar_year=YEAR, crop=CROP, **crop_management)

Printing the crop shows the details of the crop and its default management:

In [ ]:
print(crop)

Agromanagement parameters can be modified in the same way one would modify python dictionaries. The example below shows, for instance, how to modify from defaults the date in which the wofost simulation needs to start (`start_crop_calendar`). This allows for example to track water dynamics in the bare soil in the period between harvest of the previous crop and planting of the crop of interest.

In [ ]:
crop_management["start_crop_calendar"] = dt.date(YEAR, 6, 1)
print(crop_management)

The example below shows how fertilisation events are modified from default. In this case, instead of having 3 fertilisation events, we want only one on February 20th, 2020 where 250 units of nitrogen are applied.

In [ ]:
crop_management['apply_npk'] = [
    {'month': 2, 'day': 20, 'N_amount': 250, 'P_amount': 50, 'K_amount': 50}
]
print(crop_management)

Other crop management inputs can be also altered: here we change crop variety from the default one

In [ ]:
crop_management['variety'] = 'Winter_wheat_106'
print(crop_management)

While management can be customised after instantiating a crop, it is easier and recommended to customise it before a crop is initialised. Now that we have modified the crop_management dictionary, let's re-initialise the crop with custom management and verify that the custom management is applied:

In [ ]:
crop = Crop(calendar_year=YEAR, crop=CROP, **crop_management)
print(crop)

## 4. Pass the crop and management to the simulator to return yields
Once the crop and its management have been declared, we can now pass them to the simulator to run WOFOST and obtain crop yields, which is done using the `simulator.run()` method.  
This method takes two required and one optional argument. The required arguments are:
1. `crop_or_rotation`: either an instance of the `Crop` or of the `CropRotation` class;
2. `output_flag`: this determines what output is returned. The following flags are available:  
    - `summary`: this returns the crop yield at harvest in kg dry matter;
    - `full`: this returns the entire time series of the WOFOST state variables for the crop or rotation of interest.  
3. `**kwargs`: these are optional dictionary with key-value pairs for any of the parameters that need to be customised; these only include the underlying WOFOST parameters (see wofost_params class attribute), soil parameters, location parameters but not agromanagement parameters. Non-default agromanagement parameters must be modified when initialising the instance of the class 'Crop' which is then passed to this method.

The output table of the `simulator.run()` method is a standard Pandas dataframe.

In [ ]:
crop_yield = sim.run(crop_or_rotation=crop, output_flag="summary")
print(crop_yield)

Instead of individual crops, it is also possible to pass to the `simulator.run()` method a crop rotation, i.e., a succession of crops and associated agromanagement. WOFOST will run the entire rotation tracking all state variables throughout time; this is particularly useful because, when specifying rotations rather than individual crops, WOFOST can track state variables continuously also for the days in which crops are not on the ground (for instance fallow periods between consecutive crops).  
Crop rotations are built using the [`CropRotation` class](https://github.com/mcmancini/UkWofost/blob/dev/ukwofost/core/crop_manager.py#L288) which takes as input a list of instances of the class `Crop`. It is recommended to define the crop specific agromanagment when instantiating individual crops, and then combine them into a rotation afterwards.


In [ ]:
from ukwofost.core.crop_manager import CropRotation

## CROP 1
## ======
CROP_1 = "winter_wheat"
YEAR_1 = 2020

# Define management for crop 1
crop1_mgmt = copy.deepcopy(defaults.get("management").get("winter_wheat"))
crop1_mgmt["start_crop_calendar"] = dt.date(YEAR_1, 6, 1)
crop1_mgmt['apply_npk'] = [
    {'year': YEAR_1, 'month': 10, 'day': 15, 'N_amount': 250, 'P_amount': 50, 'K_amount': 50}
]

# initialise crop 1
crop_1 = Crop(calendar_year=YEAR_1, crop=CROP_1, **crop1_mgmt)

## CROP 2
## ======
CROP_2 = "spring_barley"
YEAR_2 = 2022 # Note: year after harvest of crop 1 or it would be winter barley!

# Define management for crop 2
crop2_mgmt = copy.deepcopy(defaults.get("management").get("spring_barley"))

# initialise crop 2
crop_2 = Crop(calendar_year=YEAR_2, crop=CROP_2, **crop2_mgmt)

## ROTATION
## ========
rotation = CropRotation(crops=[crop_1, crop_2])
print(rotation)

Now that we have defined a rotation, we can then pass it to the `WofostSimulator` to return crop yields for the crops in the rotation:

In [ ]:
rotation_yield = sim.run(crop_or_rotation=rotation, output_flag="full")
print(rotation_yield)